All Imports


In [17]:


import os
from typing import List, Dict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from chonkie import OpenAIEmbeddings
import time

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)


Set the API Key

In [2]:


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


Embeddings


In [3]:


embeddings = OpenAIEmbeddings()
text = "Hello Testing the embedding"
test_vector = embeddings.embed(text)

print(f"Embedding Dimension: {len(test_vector)}")
print(f"Sample embeddings: {test_vector[:5]}")


Embedding Dimension: 1536
Sample embeddings: [ 0.00168324 -0.00759125  0.03552246 -0.02740479 -0.01048279]


Reading Files


In [4]:


with open("new_data/sample_research_paper.txt", "r") as file:
    research_paper_text = file.read()

with open("new_data/sample_code.py", "r") as file:
    code_text = file.read()

with open("new_data/sample_table.md", "r") as file:
    table_text = file.read()

with open("new_data/sample_technical_doc.txt", "r") as file:
    technical_doc_text = file.read()

print(f"Technical Doc: {len(technical_doc_text):,} characters")
print(f"Research Paper: {len(research_paper_text):,} characters")
print(f"Code Sample: {len(code_text):,} characters")
print(f"Table Sample: {len(table_text):,} characters")


Technical Doc: 7,473 characters
Research Paper: 14,482 characters
Code Sample: 12,943 characters
Table Sample: 6,159 characters


Helper functions

In [ ]:
def display_chunks(chunks,max_display=3,show_metadata=True):
    """Display chunk size"""
    print(f"\n Total Lenght of chunks {len(chunks)}")
    print("*"*80)
    
    for i,chunks in enumerate(chunks[:max_display]):
        print(f"\n Chunk {i+1}: {chunks.text[:200]}....")
        if show_metadata and hasattr(chunks,'token_count'):
            if show_metadata and hasattr(chunks,'start_char'):
                print(f"Position: {chunks.start_char}-{chunks.end_char}")
        print('-'*80)
    
    if (max_display<len(chunks)):
        print(f"\n{len(chunks)-max_display} chunks left")
    

def visualize_chunk_size(chunks,title="Chunk Size Distribution"):
    token_counts=[len(chunk.token_count) if hasattr(chunk,"token_count") else len(chunk.text) for chunk in chunks]
    plt.figure(figsize=(12,8))
    plt.subplot(1, 2, 1)
    plt.hist(token_counts, bins=20, edgecolor='black', alpha=0.7)
    plt.xlabel('Token Count')
    plt.ylabel('Frequency')
    plt.title(f'{title}\nDistribution')
    plt.axvline(np.mean(token_counts), color='red', linestyle='--', label=f'Mean: {np.mean(token_counts):.0f}')
    plt.legend()
    
    # Box plot
    plt.subplot(1, 2, 2)
    plt.boxplot(token_counts, vert=True)
    plt.ylabel('Token Count')
    plt.title(f'{title}\nBox Plot')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nStatistics:")
    print(f"  Mean: {np.mean(token_counts):.1f}")
    print(f"  Median: {np.median(token_counts):.1f}")
    print(f"  Std Dev: {np.std(token_counts):.1f}")
    print(f"  Min: {min(token_counts)}")
    print(f"  Max: {max(token_counts)}")

def mpare_chunkers(chunk,chunker_dict,sample_size=1000):
    """Compare multiple chunkers at the same time"""
    sample_text= chunk[:sample_size] if len(chunk)>sample_size else text
    result={}
    for name,chunker in chunker_dict.items():
        start_time=time.time()
        chunked_text=chunker.chunk(sample_text)
        elapsed_time=time.time()-start_time
        token_counts=[chunk.token_count if hasattr(chunk,'token_count') else len(chunk.text) for chunk in chunked_text]

        result[name]={
            "num_chunks":len(chunked_text),
            "avg_size":np.mean(token_counts),
            "standard_meridian":np.std(token_counts),
            "elapsed":elapsed_time
        }
    
    df=pd.DataFrame(result).T
    return df

print("Helper Function Loaded")
